# 감성분석 · 키워드 추출 · 문서 유사도

**환경:** Google Colab / Jupyter Notebook
**준비물:** `TS_상품평_4개카테고리_통합.xlsx` (상품평 21,404건, 카테고리 4개)
**소요 시간:** 약 30분

## 오늘의 이야기
> "고객 리뷰 2만 건 중에서 **불만 리뷰를 찾고**, **무엇이 문제인지** 알아내고, **같은 불만을 가진 리뷰를 모아라!**"

| 순서 | 분석 | 질문 |
|---|---|---|
| ① | 감성분석 | 이 리뷰는 **좋다는 말일까, 나쁘다는 말일까?** |
| ② | 키워드 추출 | 불만 리뷰에서 **어떤 단어**가 많이 나올까? |
| ③ | 문서 유사도 | 이 불만과 **비슷한 리뷰**는 어떤 것이 있을까? |

## 실습 진행 과정
| 단계 | 할 일 | 핵심 메시지 |
|---|---|---|
| 준비 | 엑셀 읽기, 단어 나누기 (`안`·`못`·`않다` 남기기) | 부정어를 지우면 감성분석이 망가진다 |
| ① 감성분석 | ✏️ 긍정·부정 사전 → 점수 → 긍정/중립/부정 → 직접 채점 | 사전 방식은 간단하고 이유가 보이지만 한계도 있다 |
| ② 키워드 추출 | 부정 리뷰의 카테고리별 불만 키워드 | 어떤 리뷰를 고르느냐에 따라 보이는 것이 달라진다 |
| ③ 문서 유사도 | ✏️ 불만 문장 → 비슷한 부정 리뷰 5개 | 비슷한 불만을 모으면 반복되는 문제가 보인다 |
 | 정리 | 결과 저장, 한계 정리 | 정답이 없을 땐 **사람의 확인**이 꼭 필요하다 |

> **실행 방법:** 셀을 클릭하고 `Shift + Enter`. 반드시 **위에서부터 순서대로** 실행하세요.

---
## 0. 준비 (3분)


In [1]:
# 한국어 형태소 분석기와 엑셀 읽기 도구 설치 (처음 한 번만)
!pip install -q kiwipiepy openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 MB 8.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 107.6 MB/s eta 0:00:00


Colab 왼쪽 **📁 파일 창**에 `TS_상품평_4개카테고리_통합.xlsx`를 끌어다 놓으세요.

In [2]:
# 표 다루기 도구 불러오기
import pandas as pd

# ===== ✏️ [입력] 엑셀 파일 이름 =====
excel_path = "TS_상품평_4개카테고리_통합.xlsx"
# ===================================

# 엑셀에서 카테고리와 상품평 두 열만 읽기
df = pd.read_excel(excel_path, usecols=["카테고리", "상품평"])

# 리뷰 수 출력
print("리뷰 수:", len(df))

리뷰 수: 21404


### 단어 나누기
감성분석에서는 **부정어가 아주 중요**합니다. `좋다`와 `안 좋다`는 정반대이기 때문입니다.
그래서 명사·동사·형용사·어근과 함께 **`안`, `못`, `않다`** 를 꼭 남깁니다.

In [17]:
# Kiwi 형태소 분석기 불러오기
from kiwipiepy import Kiwi
kiwi = Kiwi()

# 남길 품사: 일반명사, 고유명사, 동사, 형용사, 어근
keep_tags = {"NNG", "NNP", "VV", "VA", "XR"}

# 형태소 분석 결과에서 필요한 단어만 골라 띄어쓰기로 이어 주는 함수
def to_words(tokens):
    words = []
    for t in tokens:
        tag = t.tag                  # 'VA' 동사 같은 태그
        if tag in {"VV", "VA"} or t.form == "않":
            words.append(t.form + "다")            # 동사·형용사·'않'은 '다' 붙이기
        elif tag == "XR":
            words.append(t.form + "하다")          # 어근은 '하다' 붙이기 (촉촉 → 촉촉하다)
        elif tag in keep_tags or t.form in {"안", "못"}:
            words.append(t.form)                  # 명사와 부정어 '안', '못'은 그대로
    return " ".join(words)

# 리뷰 전체를 한 번에 분석해서 '단어' 열 만들기 (20초 정도 걸림)
df["단어"] = [to_words(tokens) for tokens in kiwi.tokenize(df["상품평"].tolist())]

# 원문과 단어 비교
df[["상품평", "단어"]].head()

,상품평,단어
0,남편 사 줬는데 광고에서 말한 것처럼 향이 은은하지 않고 올드한 남자 스킨로션 향이...,남편 사다 광고 말 향 은은하다 않다 올드 남자 스킨 로션 향 향
1,역시 전에 쓰던 게 좋은 것 같아요.\n향도 독하고 피부에 자극이 강해서 별로예요.,전 쓰다 좋다 같다 향 독하다 피부 자극 강하다
2,남편 선물로 샀는데 저렴하고 향기좋아서 지인들에게 꼭 추천하고 싶어요~~♡♡\n아참...,남편 선물 사다 저렴하다 향기 좋다 지인 추천 택배 상자 안 상자 찌그러지다 오다 ...
3,"스킨 2개, 로션 1개 구성으로 3세트\n가성비도 좋고 항상 스킨이 부족하였는데 ...",스킨 로션 구성 세트 가성비 좋다 스킨 부족 구성 넉넉하다 좋다 향 좋다 부드러
4,제일 좋아하는 화장품이고 가장 효과가 좋은 화장품입니다. 제형의 농도가 짙고 착 감...,좋아하다 화장품 효과 좋다 화장품 형 농도 짙다 감기다 끈적이다 없다 흡수 좋다


---
## ① 감성분석

### 🎯 목표
리뷰마다 **긍정 / 중립 / 부정**을 정합니다.

### 💡 방법: 점수 매기기 (감성 사전)
1. **긍정 단어** 목록과 **부정 단어** 목록을 만듭니다. (= 감성 사전)
2. 리뷰에 긍정 단어가 나오면 **+1**, 부정 단어가 나오면 **−1**
3. 단, 바로 앞에 `안`·`못`이 있거나 바로 뒤에 `않다`·`없다`가 오면 **부호를 뒤집습니다.**
4. 점수 합계가 **0보다 크면 긍정, 0이면 중립, 0보다 작으면 부정**

| 리뷰 단어 | 계산 | 점수 |
|---|---|---|
| 배송 **빠르다** 포장 **깔끔하다** | +1 +1 | **+2 → 긍정** |
| 화면 **선명** 소리 **별로** | +1 −1 | **0 → 중립** |
| 품질 **안** **좋다** | `안` 때문에 +1 → −1 | **−1 → 부정** |
| **불편** **없다** | `없다` 때문에 −1 → +1 | **+1 → 긍정** |

> 💡 이 방식은 **정답 데이터 없이** 쓸 수 있고, 왜 그런 결과가 나왔는지 **이유가 보인다**는 장점이 있습니다.

**✏️ [입력]** 결과를 보고 사전에 단어를 더하거나 빼 보세요.

In [30]:
# ===== ✏️ [입력] 감성 사전 =====
# 긍정 단어 집합(set): 중괄호 { } 안에 문자열을 나열해 중복 없는 집합을 만듦
pos_words = {"좋다", "만족", "최고", "저렴하다", "편하다", "빠르다", "깔끔하다", "튼튼하다", "선명",
             "추천", "괜찮다", "촉촉하다", "은은하다", "편리", "가성비", "훌륭하다", "간편하다", "쉽다"}
# 부정 단어 집합(set)
neg_words = {"별로", "아쉽다", "불편", "비싸다", "느리다", "불만", "실망", "고장", "최악",
             "나쁘다", "귀찮다", "어렵다", "답답하다", "약하다", "끈적이다", "소음", "불량"}
# ==============================

# 리뷰 하나(공백으로 구분된 토큰 문자열)를 받아 감성 점수(정수)를 돌려주는 함수
def sentiment_score(text):
    words = text.split()                  # 공백 기준으로 나눠 단어 리스트를 만듦
    score = 0                             # 누적 점수를 0에서 시작
    for i, w in enumerate(words):         # i = 단어 위치 번호, w = 그 위치의 단어
        if w in pos_words:                # 단어가 긍정 집합에 있으면
            value = 1                     #   이 단어의 점수를 +1로 정함
        elif w in neg_words:              # 긍정은 아니고 부정 집합에 있으면
            value = -1                    #   이 단어의 점수를 -1로 정함
        else:                             # 두 집합 어디에도 없으면
            continue                      #   아래 코드를 건너뛰고 다음 단어로 넘어감
        # 앞 단어: i가 0보다 크면 words[i-1], 첫 단어면 빈 문자열
        before = words[i - 1] if i > 0 else ""
        # 뒤 단어: 다음 위치가 리스트 범위 안이면 words[i+1], 마지막 단어면 빈 문자열
        after = words[i + 1] if i + 1 < len(words) else ""
        # 앞 단어가 '안' 또는 '못'이거나, 뒤 단어가 '않다' 또는 '없다'이면
        if before in {"안", "못"} or after in {"않다", "없다"}:
            value = -value                # 이 단어 점수의 부호를 바꿈 (+1 → -1, -1 → +1)
        score += value                    # 이 단어 점수를 누적 점수에 더함
    return score                          # 리뷰 전체 점수를 돌려줌

# 점수(정수)를 받아 '긍정'/'부정'/'중립' 문자열을 돌려주는 함수
def to_label(score):
    if score > 0:                         # 0보다 크면
        return "긍정"                     #   '긍정'을 돌려주고 함수 종료
    if score < 0:                         # 0보다 작으면
        return "부정"                     #   '부정'을 돌려주고 함수 종료
    return "중립"                         # 위 두 조건이 모두 아니면(= 0) '중립'

# '단어' 열의 각 값에 sentiment_score를 적용해 '점수' 열을 새로 만듦
df["점수"] = df["단어"].apply(sentiment_score)
# '점수' 열의 각 값에 to_label을 적용해 '감성' 열을 새로 만듦
df["감성"] = df["점수"].apply(to_label)

# value_counts(normalize=True): 값별 비율(0~1) → ×100: 백분율 → round(1): 소수 첫째 자리 반올림
#normalize=True면 개수 대신 **상대 빈도(비율)**를 돌려줍니다.
#(df["감성"].value_counts() * 100).round(1) #전체갯수
print( (df["감성"].value_counts(normalize=True) * 100).round(1) )


감성
긍정    77.6
중립    14.2
부정     8.3
Name: proportion, dtype: float64


In [31]:
# 카테고리별 감성 비율(%) 표
# 1단계: 행 = 카테고리, 열 = 감성, 값 = 리뷰 개수인 교차표를 만듦
#        normalize="index": 각 행(카테고리)의 값을 그 행의 합계로 나눠 행마다 합이 1이 되게 함
ratio = pd.crosstab(df["카테고리"], df["감성"], normalize="index")

# 2단계: 0~1 비율에 100을 곱해 백분율로 바꿈
percent = ratio * 100

# 3단계: 소수 첫째 자리까지 반올림
print(percent.round(1))

감성         긍정    부정    중립
카테고리                     
남성화장품    74.2  13.9  11.9
영상/음향가전  88.2   3.6   8.2
자동차기기    71.2  11.2  17.6
청소/세탁용품  77.0   5.4  17.6


### 결과 눈으로 확인하기
감성별로 리뷰를 3개씩 봅니다. 점수가 맞게 나왔나요?

In [32]:
# 감성별로 리뷰 3개씩 출력
for label in ["긍정", "중립", "부정"]:
    print(f"===== {label} =====")
    sample = df[df["감성"] == label].sample(3, random_state=1)
    for _, row in sample.iterrows():
        print(f"[{row['점수']:+d}] {row['상품평'][:80]}")
    print()

===== 긍정 =====
[+2] 저전력 모드 기능이 있어서 너무 만족스럽게 쓰고 있습니다~ 착한 가격은 덤이에요 지인들한테 추천중^^
[+3] 한달이상 주차 녹화 가능한 초저전력이 최고인것 같아요. GPS 포함에 차선이탈기능까지 너무 좋아요! 화질도 깨끗해서 마음에 들구요. 배터리 방전
[+1] 처음 사봤는데 엄청 좋습니다. 머리카락 청소하려고 샀는데 사용하기 너무 편합니다. 조금 약한것 같은데 사용하는데 무리는 없네요

===== 중립 =====
[+0] 기존에 사용하던 스킨 로션은 여름철에 유분기가 많더라고요. 그래서 이 제품을 주문해 봤습니다. 두달 간 사용해 봤는데 이 제품도 여름철에 사용하
[+0] 화면 사이즈가 8인치로 상당히 커서 지도 볼때 시원해서 좋네요.. 
기기가 전체적으로 탄탄하지 않고 약해보여서 별로에요. ㅜ 아쉽습니다.
[+0] 향은 남성 화장품 냄새가 좀 강해서 아쉽고 유통기한은 내년 3월까지라 무지 짧네요.

===== 부정 =====
[-2] 화면화질이 생각보다 선명하지 않아서 조금 실망이네요 그리고 LED밝기가 환하지 않아서 별로네요
[-1] 카드가 꽃혀있는대도 불고하고 카드 오류라고 나와서 카드단말기 불량이네요 ㅠㅠ 
지문인식도 제대로 안되고 뭔가요 ㅠㅠ
[-1] 조작이 화면을 터치하는게 아니라 버튼을 눌러야 돼서 불편합니다. 또 경보 기능들이 오류가 좀 있네요.



### ✍️ 직접 채점하기
이 데이터에는 **정답이 없어서** 컴퓨터가 정확도를 계산할 수 없습니다. 그래서 **사람이 직접 확인**해야 합니다.

1. 아래 셀을 실행해 무작위 리뷰 10개를 봅니다.
2. 각 리뷰를 읽고 내 판단을 `my_labels`에 순서대로 적습니다. (`"긍정"`, `"중립"`, `"부정"`)
3. 다음 셀을 실행하면 사전 결과와 내 판단이 몇 개 일치하는지 나옵니다.

In [33]:
# 무작위 리뷰 10개 뽑기 (매번 같은 10개)
check = df.sample(10, random_state=42)

# 번호와 리뷰 출력 (사전 결과는 숨김)
for n, text in enumerate(check["상품평"], 1):
    print(n, text[:100])

1 뒷면에 C타입 단자가 있어 케이블 하나로 전원등 모든게 연결이 되어 아주 쉽고 편해요. 사용중인 사운드바와 비교해 보니 음질 차이는 잘 모르겠고 그럭저럭 무난하게 들을만해요.
2 옷 휙휙 벗어던지는 애들때문에 구매했다. 정리잘해서 벗어 놓았네요 조립이 조금 힘들었지만 디자인은 깔끔해서 만족합니다
3 전에 쓰던게 다리가 고장나서 같은거 주문했습니다 빨래가 많이 널려서 좋습니다. 생각보다 약한게 단점입니다
4 제공되는 리모컨이 일반 리모컨이라서 셋탑박스하고는 연동이 안 되는 점이 불편하네요 ㅠㅠ
그래도 색상이 깔끔해서 방이랑 잘어울려 좋아요
5 22년식 최신 내비라 아주 좋네요 굿굿
매립방법 간단해서 아주 좋고 잘 쓰고 다녔습니다^^
6 이전에 구매해서 쓰던 제품이라 불만없어요. 사이즈 컴팩트해서 어디든 부착하기 좋아요 유선이라 선이 깔끔하지 않는건 있어요 ㅠㅠ
근데 받기도 전에 배송완료 해놓고 언제까지 기다려야 
7 일단 화질이 넘 선명해서 좋고 내비게이션 경로탐색 기능도 우수해  길도 잘 찾아줘요
8 진짜 저렴한 가격에 잘 사긴 했는데 배송이 무려 한달이나 걸렸어요 ㅠㅠ... 그것만 빼고 티비 자체는 기능 완전 최고에요!
9 접착력도 좋고 길어서 좋고, 참 편리합니다. 장롱 옮기면서 먼지 많은 곳에 청소기 쓰기 힘들었는데 이걸로 먼지 닦아내고 마무리로 청소기 돌렸네요 건식화장실 머리카락도 닦어내니 좋아
10 해외 물류 상황에 따라 배송기간이 길어진점은 아쉽네요
그래도 스피커 상태는 흠집없이 양호한 상태라 아주 좋습니다
스피커 어디든 호환되고 출력기능이 좋아서 맘에 드네요


In [37]:
# ===== ✏️ [입력] 위 10개 리뷰에 대한 내 판단 (순서대로 10개) =====
my_labels = []
my_labels = ["긍정", "긍정", "부정", "중립", "긍정", "긍정", "부정", "긍정", "긍정", "중립"]
# ================================================================

# 10개를 모두 적었으면 일치 개수 계산
if len(my_labels) == 10:
    check["내 판단"] = my_labels
    agree = (check["내 판단"] == check["감성"]).sum()
    print(f"사전 결과와 내 판단 일치: {agree} / 10")
    display(check[["상품평", "감성", "내 판단"]])
else:
    print("my_labels에 판단 10개를 적은 뒤 다시 실행하세요.")

사전 결과와 내 판단 일치: 7 / 10


,상품평,감성,내 판단
6178,뒷면에 C타입 단자가 있어 케이블 하나로 전원등 모든게 연결이 되어 아주 쉽고 편해...,긍정,긍정
21230,옷 휙휙 벗어던지는 애들때문에 구매했다. 정리잘해서 벗어 놓았네요 조립이 조금 힘들...,긍정,긍정
20997,전에 쓰던게 다리가 고장나서 같은거 주문했습니다 빨래가 많이 널려서 좋습니다. 생각...,부정,부정
8294,제공되는 리모컨이 일반 리모컨이라서 셋탑박스하고는 연동이 안 되는 점이 불편하네요 ...,긍정,중립
9512,22년식 최신 내비라 아주 좋네요 굿굿\n매립방법 간단해서 아주 좋고 잘 쓰고 다녔...,긍정,긍정
13478,이전에 구매해서 쓰던 제품이라 불만없어요. 사이즈 컴팩트해서 어디든 부착하기 좋아요...,긍정,긍정
10319,일단 화질이 넘 선명해서 좋고 내비게이션 경로탐색 기능도 우수해 길도 잘 찾아줘요,긍정,부정
4772,진짜 저렴한 가격에 잘 사긴 했는데 배송이 무려 한달이나 걸렸어요 ㅠㅠ... 그것만...,긍정,긍정
18830,"접착력도 좋고 길어서 좋고, 참 편리합니다. 장롱 옮기면서 먼지 많은 곳에 청소기 ...",긍정,긍정
6824,해외 물류 상황에 따라 배송기간이 길어진점은 아쉽네요\n그래도 스피커 상태는 흠집없...,긍정,중립


🤔 **생각해 볼 점**
- 장점과 단점이 **섞인 리뷰**(예: "만족하는데 지문인식은 별로")는 점수가 0이 되어 **중립**으로 나옵니다. 맞는 결과일까요?
- `작다`는 블랙박스에선 장점(작아서 좋음), TV에선 단점(화면이 작음)입니다. 그래서 사전에 넣지 않았습니다. **카테고리마다 사전을 따로 만들어야 할까요?**
- "좋을 줄 알았는데 아니네요", 반어법("참 빠르네요… 일주일 걸림")은 이 규칙으로 잡지 못합니다.
- 같은 말이 **다른 모양**으로 나뉘기도 합니다. `깔끔하면서도`는 문맥에 따라 `깔끔`(명사)으로 분석되어 사전의 `깔끔하다`와 맞지 않습니다. 사전에 두 모양을 함께 넣어야 할 수 있습니다.
- "귀찮아하는 남편 때문에 샀는데 좋아요"처럼 **제품이 아닌 대상**에 대한 부정어도 점수에 들어갑니다.
- 일치 개수 10개는 **통계적으로 믿기엔 너무 적습니다.** 실무에서는 수백 개를 사람이 확인합니다.

---
## ② 키워드 추출

### 🎯 목표
**부정 리뷰만** 모아서, 카테고리별로 **무엇이 불만인지** 키워드를 뽑습니다. 긍정 리뷰 키워드와 비교합니다.

### 💡 방법
단원 3의 TF-IDF를 씁니다. 카테고리별로 리뷰들의 TF-IDF 점수 **평균이 높은 단어**가 키워드입니다.
감성 단어(`별로`, `좋다` 등)는 이미 감성으로 나눴으니 키워드에서 빼고 **"무엇이"** 좋고 나쁜지에 집중합니다.

In [38]:
# TF-IDF 도구 불러오기
from sklearn.feature_extraction.text import TfidfVectorizer

# 키워드에서 뺄 단어: 감성 사전 단어 + 뜻이 약한 단어
skip = pos_words | neg_words | {"하다", "되다", "있다", "같다", "들다", "사용", "제품", "구매", "쓰다", "안", "못", "않다", "없다"}

# 공통 설정: 띄어쓰기로만 나누기, 대소문자 유지, skip 단어 빼기
opts = dict(tokenizer=str.split, token_pattern=None, lowercase=False, stop_words=list(skip))

# 리뷰 묶음을 받아 카테고리별 키워드 10개 표를 돌려주는 함수
def category_keywords(sub):
    tv = TfidfVectorizer(min_df=3, **opts)               # 문서빈도가 3보다 작은 단어를 사전에서 제외하는 TF-IDF 도구
    X = tv.fit_transform(sub["단어"])                     # 사전·IDF를 학습하고 리뷰를 TF-IDF 희소 행렬로 바꿈

    # 1단계: TF-IDF 점수를 표로 만들기 (행 = 리뷰, 열 = 단어)
    scores = pd.DataFrame(X.toarray(), columns=tv.get_feature_names_out())

    # 2단계: 각 리뷰의 카테고리를 열로 붙이기
    # .values: sub의 행 번호(index)는 무시하고 순서대로 값만 붙임
    scores["카테고리"] = sub["카테고리"].values

    # 3단계: 카테고리별로 묶어 단어마다 평균 점수 계산 (행 = 카테고리, 열 = 단어)
    # sort=False: 카테고리를 데이터에 처음 나온 순서대로 둠
    avg = scores.groupby("카테고리", sort=False).mean()

    # 4단계: 카테고리마다 평균 점수가 큰 단어 10개 고르기
    table = {}                                           # {카테고리: 상위 단어 10개}를 담을 딕셔너리
    for cat in avg.index:                                # avg의 행 이름(카테고리)을 하나씩 꺼냄
        top10 = avg.loc[cat].sort_values(ascending=False).head(10)   # 그 카테고리 행을 큰 값부터 정렬해 앞 10개
        table[cat] = top10.index                         # 값이 아니라 단어 이름(index)만 저장
    return pd.DataFrame(table)                           # 열 = 카테고리, 행 = 1~10위 단어인 표로 돌려줌


In [39]:
# 부정 리뷰만 고르기
neg = df[df["감성"] == "부정"]

# 카테고리별 부정 리뷰 수 (적으면 키워드가 불안정할 수 있음)
print(neg["카테고리"].value_counts())

# 부정 리뷰 카테고리별 키워드
category_keywords(neg)

카테고리
자동차기기      898
남성화장품      417
청소/세탁용품    239
영상/음향가전    214
Name: count, dtype: int64


,남성화장품,영상/음향가전,자동차기기,청소/세탁용품
0,향,티비,화질,빨래
1,피부,음질,속도,널다
2,바르다,소리,설치,조립
3,가격,연결,화면,생각
4,사다,크다,인식,가격
5,보습,스피커,나다,부분
6,처음,가격,기능,바퀴
7,품질,사운드,연결,보이다
8,배송,나다,보이다,사다
9,발림,화질,부팅,크다


In [40]:
# 비교용: 긍정 리뷰 카테고리별 키워드
category_keywords(df[df["감성"] == "긍정"])

,남성화장품,영상/음향가전,자동차기기,청소/세탁용품
0,향,음질,화질,빨래
1,바르다,소리,가격,청소
2,피부,화질,설치,널다
3,인원,가격,기능,조립
4,가격,사운드,인식,가격
5,배송,디자인,화면,배송
6,남편,배송,마음,사다
7,올,크다,보이다,생각
8,보습,보다,디자인,크다
9,로션,연결,속도,건조대


✅ **결과 확인:** 부정 키워드에는 고장·소음·크기처럼 **문제점**이, 긍정 키워드에는 화질·향·배송처럼 **좋은 점**이 보이나요?
같은 TF-IDF라도 **어떤 리뷰를 골라 넣느냐**에 따라 보이는 것이 완전히 달라집니다.

---
## ③ 문서 유사도

### 🎯 목표
고객 불만 하나를 입력하면 **비슷한 부정 리뷰 5개**를 찾습니다. 같은 문제가 반복되는지 확인하는 실무 방법입니다.

### 💡 방법
부정 리뷰로 TF-IDF를 만들고, 불만 문장도 같은 방식으로 바꾼 뒤 **코사인 유사도**로 비교합니다.

**✏️ [입력]** `complaint`에 불만 문장을 넣어 보세요. (예: `화면이 안 켜져요`, `조립이 어려워요`, `향이 너무 강해요`)

In [41]:
# 유사도 계산 도구 불러오기
from sklearn.metrics.pairwise import cosine_similarity

# ===== ✏️ [입력] 불만 문장 =====
complaint = "소리가 작고 연결이 자꾸 끊겨요"
# ==============================

# 부정 리뷰로 TF-IDF 만들기 (감성 단어는 빼지 않고 그대로 사용)
tv_neg = TfidfVectorizer(tokenizer=str.split, token_pattern=None, lowercase=False)
X_neg = tv_neg.fit_transform(neg["단어"])

# 불만 문장을 단어로 나누고 같은 TF-IDF로 바꾸기
complaint_vec = tv_neg.transform([to_words(kiwi.tokenize(complaint))])

# 불만 문장과 모든 부정 리뷰의 유사도 계산
scores = cosine_similarity(complaint_vec, X_neg)[0]

# 유사도가 높은 부정 리뷰 5개 출력
result = neg.assign(유사도=scores).sort_values("유사도", ascending=False)
result[["유사도", "카테고리", "상품평"]].head(5)

,유사도,카테고리,상품평
8408,0.412322,영상/음향가전,호환성이 매우 떨어져서 컴퓨터랑 블루투스로 연결하면 금방 끊겨서 너무 불편해요. 다...
5420,0.398667,영상/음향가전,생각보다 사이즈가 커서 별루예요 그리고 인터넷방송이나 블루투스 연결할떄 소리가 끊겨...
10996,0.387727,자동차기기,DMB 소리가 작아도 너무 작아서 실망이 큽니다. \n또 저장 용량이 너무 작아서 ...
9840,0.376179,자동차기기,무선 연결 기능 진짜 꽝!! 연결이 자꾸 끊기고 심지어 안될 때가 더 많음.. 터치...
11788,0.366662,자동차기기,음성 안내소리가 작아 볼륨을 키워도 차 소음떄문에 작게 들려서 아쉬워요 화면이 작아...


✅ **결과 확인:** 비슷한 불만이 같은 카테고리에서 반복되나요? 이렇게 모은 리뷰는 **고객 응대·제품 개선**에 바로 쓸 수 있습니다.

---
## 정리

In [42]:
# 결과(원문, 카테고리, 점수, 감성)를 CSV 파일로 저장 (한글 깨짐 방지 인코딩)
df[["카테고리", "상품평", "점수", "감성"]].to_csv("sentiment_result.csv", index=False, encoding="utf-8-sig")
print("저장 완료: sentiment_result.csv")

저장 완료: sentiment_result.csv


### ✅ 오늘의 한 줄 정리
| 분석 | 한 줄 정리 |
|---|---|
| 감성분석 | 긍정·부정 사전으로 점수를 매기고, 부정어가 있으면 부호를 뒤집는다 |
| 키워드 추출 | **부정 리뷰만** 골라 TF-IDF로 보면 불만의 원인이 보인다 |
| 문서 유사도 | 불만 문장과 비슷한 리뷰를 모아 반복되는 문제를 찾는다 |

### ⚠️ 꼭 기억할 한계
- 이 감성 결과는 **검증된 정답이 아닙니다.** "긍정 70%"처럼 보고할 때는 사람이 확인한 샘플의 일치율을 함께 밝혀야 합니다.
- 사전에 없는 표현, 섞인 감정, 반어법에 약합니다.
- 정답 라벨이 있는 데이터가 생기면, 단원 4의 **분류 모델**로 감성을 학습할 수 있습니다.



**참고**
- [scikit-learn TfidfVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) · [cosine_similarity](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html)
- 심화: [KNU 한국어 감성사전](https://github.com/park1200656/KnuSentiLex) (약 1만 4천 개 항목, 저장소에 라이선스가 명시되어 있지 않으니 배포·활용 전 확인 필요)